# User input to tool calls

One question runs through the agents. Each cell shows one step.

`user input -> plan -> KG reading -> figure reading -> design -> tool calls`

Section 1 holds the tools as stand-ins. Wire the final tools there. Section 2 ties a bioinspired mechanism from the KG to a printed ceramic.

Run from the repo root with the model server up. Needs `jupyter`.

In [ ]:
import json
import re
import time
from IPython.display import Image

from harness import corpus, graph, tools as standin
from harness.llm import parse
from harness.tools import material, number, spec

c = corpus.load("examples/corpus", graph.EMBEDDING)  # 8 papers as passages, figures and KG triples
figures = int(c["is_figure"].sum())
print(len(c["chunks"]) - figures, "passages,", figures, "figures,", len(c["triples"]), "triples")
graph.MODELS  # model and reasoning effort per agent

## 1. Tools

Embed the final tools here. Keep the input and output names, or change them in the schema too.

In [ ]:
def dft_cleavage(material):
    # Wire to final DFT tool. Stand-in is a surface-energy table.
    return {"cleavage_energy_J_m2": 2 * standin.SURFACE_ENERGY[material],
            "method": "stand-in (table of surface energies), not a DFT result"}


def md_interface(material_a, material_b):
    # Wire to final MD tool. Stand-in is the closed form in harness/tools.py.
    return standin.md_interface(material_a, material_b)


def phase_field_fracture(layer_modulus_GPa, layer_thickness_um, layer_fracture_energy_J_m2,
                         interface_fracture_energy_J_m2, interface_strength_MPa):
    # Wire to final phase-field tool. Stand-in is the He-Hutchinson criterion.
    ratio = interface_fracture_energy_J_m2 / layer_fracture_energy_J_m2
    # AT2 length that matches the interface strength, l = 27 E G / (256 s^2)
    length = 27 * layer_modulus_GPa * 1e9 * interface_fracture_energy_J_m2 / (256 * (interface_strength_MPa * 1e6) ** 2)
    return {"crack_deflects": ratio < 0.25, "energy_ratio": round(ratio, 3),
            "criterion": "deflects when energy_ratio < 0.25 (He-Hutchinson, elastically similar layers)",
            "interface_length_scale_nm": round(length * 1e9, 2),
            "method": "stand-in (He-Hutchinson criterion), not a phase-field result"}

In [ ]:
# Tool schemas the engineer reads. Each input names its source.
graph.TOOLS = [
    spec("dft_cleavage",
         "Density-functional-theory energy to cleave a material. Returns the cleavage energy in J/m^2.",
         {"material": material | {"description": "the ceramic of the design"}}),
    spec("md_interface",
         "Molecular dynamics separation of the interface between two materials at 300 K. "
         "Returns the separation energy in J/m^2 and the strength in MPa.",
         {"material_a": material | {"description": "the ceramic of the design"},
          "material_b": material | {"description": "the interlayer of the design"}}),
    spec("phase_field_fracture",
         "Phase-field crack propagation through a layered ceramic. Reports whether a crack deflects "
         "along the interfaces.",
         {"layer_modulus_GPa": number("from the design"),
          "layer_thickness_um": number("from the design"),
          "layer_fracture_energy_J_m2": number("cleavage_energy_J_m2 of dft_cleavage"),
          "interface_fracture_energy_J_m2": number("separation_energy_J_m2 of md_interface"),
          "interface_strength_MPa": number("strength_MPa of md_interface")}),
]
graph.RUN = {"dft_cleavage": dft_cleavage, "md_interface": md_interface, "phase_field_fracture": phase_field_fracture}

## 2. Bioinspired mechanism and print process

Three more design fields. The designer picks each from the KG and the papers, with a source. The KG is asked about every option, and it ranks first the triples that link two options, such as nacre and crack deflection.

In [ ]:
def choice(options, description):
    return {"type": "string", "enum": options, "description": description}


graph.DESIGN = {
    "biological_model": choice(["nacre", "bone", "Bouligand"], "the biological material the design copies"),
    "mechanism": choice(["crack deflection", "crack bridging", "compressive residual stress"],
                        "the toughening mechanism of that biological material, as the evidence states"),
    "process": choice(["direct ink writing", "stereolithography", "tape casting", "binder jetting"],
                      "the additive manufacturing process that builds the ceramic layers, as the evidence shows "
                      "for this ceramic"),
} | graph.DESIGN
graph.DESIGN["layer_thickness_um"] = number("thickness of one ceramic layer, within the range the evidence "
                                            "reports for the chosen process")
# The agents read the design fields through these two names.
graph.FIELDS = json.dumps(graph.DESIGN)
graph.OPTIONS = sorted({o for field in graph.DESIGN.values() for o in field.get("enum", ())})
list(graph.DESIGN)

## 3. User input

`state` holds what every agent reads and writes.

In [ ]:
question = ("Design a nacre-inspired layered alumina made by additive manufacturing. Choose the toughening "
            "mechanism to copy from nacre, a print process, an interlayer and a layer thickness, and simulate the design.")
app = graph.build(graph.MODELS, c)
start = time.perf_counter()
for mode, chunk in app.stream({"question": question}, {"recursion_limit": 40}, stream_mode=["updates", "values"]):
    if mode == "updates":
        print(f"{time.perf_counter() - start:6.1f} s  {', '.join(chunk)}")
    else:
        state = chunk

### Plan
Search queries and KG entities from the question.

In [ ]:
parse(state["trace"][0]["reply"]["content"])

### KG reading
Retrieved passages and figures, plus KG triples for the entities and every design option. The triples listed link two options. They carry the mechanism from the biological model to the ceramic.

In [ ]:
passages = [p for p in state["passages"] if "image" not in p]
print(len(passages), "passages,", len(state["passages"]) - len(passages), "figures,", len(state["triples"]), "triples")
print("\nPassages")
for p in passages[:3]:
    print(" ", p["id"], "|", " ".join(p["text"][:110].split()))

options = [o.lower() for o in graph.OPTIONS]
print("\nKG triples that link two options")
for t in state["triples"]:
    hit = [o for o in options if o in f"{t['head']} {t['relation']} {t['tail']}".lower()]
    if len(hit) > 1:
        print(" ", hit, "|", t["relation"][:95], "|", t["source"])

### Figure reading
Values the vision model read from each figure. The figures are CC BY 4.0, credited in `examples/corpus/ATTRIBUTION.md`.

In [ ]:
note = max(state["figure_notes"], key=lambda n: len(n["reading"].get("values", [])))
image = next(p["image"] for p in state["passages"] if p["id"] == note["id"])
display(Image(filename=str(c["folder"] / image), width=420))
print(note["id"], "|", note["reading"].get("summary"))
print()
for n in state["figure_notes"]:
    print(n["id"], [(v.get("quantity"), v.get("value"), v.get("unit")) for v in n["reading"].get("values", [])])

### Design
One value per field, each with its source. The first three fields are the bioinspired mechanism and the print process. A field with no evidence stops the run before any tool call.

In [ ]:
for field in state["design"]:
    print(f"{field:20} {str(state['design'][field]):14} {state['sources'].get(field, '')[:90]}")
print("missing:", state["missing"])

No agent checks a source. This cell screens each one by text match. A paraphrase or a converted unit gives no match, so those need a check by hand.

In [ ]:
# Source check by text match. A paper id without # means any evidence from that paper.
norm = lambda s: s.replace("/", "_")
evidence = [(p["id"], p["text"]) for p in state["passages"] if "image" not in p]
evidence += [(f["id"], json.dumps(f["reading"])) for f in state["figure_notes"]]
evidence += [(norm(t["source"]), f"{t['head']} {t['relation']} {t['tail']}") for t in state["triples"]]
evidence += [("question", question)]
for field, value in state["design"].items():
    cited = norm(state["sources"][field].split(":")[0].strip("[] "))
    text = " ".join(x for i, x in evidence if i == cited or ("#" not in cited and i.split("#")[0] == cited))
    found = re.search(rf"(?<![\d.]){re.escape(str(value))}(?![\d.])", text, re.I)
    print(f"{field:20} {str(value):20} {'match' if found else 'check by hand'}   {cited}")

In this run both numbers need the check. The cited passage does not contain 380 GPa. The 40 µm is a stereolithography layer from another paper, and the chosen process is direct ink writing.

### Tool calls
The engineer's calls, in order. Every argument comes from the design or an earlier tool output. The final tools receive these same calls.

In [ ]:
for r in state["results"]:
    print(f"{r['tool']}({json.dumps(r['arguments'])})\n   -> {r['output']}\n")

## What a final tool must match

- **Inputs.** Keyword arguments with the unit in the name, such as `layer_modulus_GPa`.
- **Output.** A small dictionary of named numbers plus a `method` string. Large results go to files, and the dictionary returns the paths.
- **Errors.** Raise an exception. The engineer sees the message and retries.
- **Run time.** The graph waits for the function. A long run should submit a job and poll.

See `docs/integration.md` for the contracts and the open points.